# Phase 2D-D | Protocol-aware light-train reconstruction and electrical-confound audit

**Research question.** Can DANDI 000005 support reliable, independent-animal optical-response modeling after grouping repeated 5-ms pulses into complete stimulation trains and checking concurrent injected current?

**Data:** [DANDI 000005, published version `0.220126.1853`](https://dandiarchive.org/dandiset/000005/0.220126.1853). Five previously audited mouse recordings (pinned file identifiers).

**Important safeguards:**

- This study uses **real mouse intracellular recordings**, not human epilepsy data.
- A positive-valued `OptogeneticSeries` is a **recorded command**, not independent proof of delivered light or a physically calibrated power value.
- Pulse trains, not individual pulses, are the **analysis units**. Repeated trains within one mouse are not independent animals.
- Time overlap does not independently prove perfect physical synchronization.
- Voltage changes can reflect injected current, protocol differences, drifts, or other confounding. **This is an exploratory audit, not causal proof or a therapy study.**
- **No DL model is trained in this notebook.** Only proceed after the final gate is satisfied.

## Kaggle instructions

1. Upload this `.ipynb` to Kaggle, **Internet = ON**, **Accelerator = None**. No additional Kaggle input required.
2. Run all cells. Five NWB assets total approximately **402 MB**; files are downloaded one-by-one.
3. Download `phase2dd_protocol_aware_results.zip` from `/kaggle/working` and upload it for review.
4. Keep `TRAIN_GAP_MS = 200` for the primary train definition; the code also reports sensitivity at other gaps.

The notebook uses the same five subject folders verified in Phase 2D-C and checks pinned assets against the published DANDI manifest. The ZIP contains tables and real-data figures, **not raw NWB data**.

In [1]:
import json, math, time, zipfile, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import h5py, requests
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

DANDI_ID, VERSION = '000005', '0.220126.1853'
API = 'https://api.dandiarchive.org'
OUT = Path('/kaggle/working/phase2dd_outputs') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2dd_outputs')
CACHE = OUT / 'temporary_nwb'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

TRAIN_GAP_MS = 200.0
GAP_SENSITIVITY_MS = [20,30,50,100,150,200,250,400]
MAX_DOWNLOAD_MB = 450.0
MAX_PER_FILE_MB = 125.0
MAX_SERIES_SAMPLES = 4_000_000
MAX_PULSES_PER_SERIES = 5000
SHAM_PER_TRAIN = 30
SHAM_MAX_ATTEMPTS_PER_TRAIN = 600
MIN_SHAM_DISTANCE_S = 0.5
RANDOM_SEED = 42
DELETE_RAW_AFTER_ANALYSIS = True
rng = np.random.default_rng(RANDOM_SEED)
ERRORS=[]

HTTP = requests.Session()
HTTP.mount('https://', HTTPAdapter(max_retries=Retry(total=3, connect=3, read=2,
                backoff_factor=1.0, status_forcelist=[429,500,502,503,504], allowed_methods=['GET'])))
HTTP.headers['User-Agent'] = 'Phase2DD-protocol-aware-neurophysiology-audit/1.0'
print('Phase 2D-D | DANDI', DANDI_ID, VERSION, '| CPU, Internet required')

Phase 2D-D | DANDI 000005 0.220126.1853 | CPU, Internet required


## 1. Verify fixed, independent-animal recordings

These exact assets were selected and processed successfully in Phase 2D-C. We pin file IDs to avoid silently swapping experiments between notebook runs.

In [2]:
PINNED = [
    ('sub-anm266945', '3ee6887c-1462-4d39-a3f3-e0e356e673d5', 46543184),
    ('sub-anm266951', 'b73da40b-a5bf-4f1c-9cfc-479b1ea4d0f3', 58123096),
    ('sub-anm281592', '4ff4ef0c-775e-491e-a3a3-246891335a86', 78389912),
    ('sub-anm244025', 'fbc7ba70-d295-44a4-bcbe-77a8a4371f15', 104930672),
    ('sub-anm260073', 'cce37384-3485-44ed-ae57-ef11142e0ee8', 113990856),
]

def fetch_manifest():
    url=f'{API}/api/dandisets/{DANDI_ID}/versions/{VERSION}/assets/?page_size=1000'
    data=[]
    for _ in range(50):
        r=HTTP.get(url,timeout=(20,90));r.raise_for_status()
        j=r.json(); data.extend(j.get('results',[])); url=j.get('next')
        if not url: break
    else: raise RuntimeError('Manifest pagination limit exceeded')
    return pd.DataFrame(data)

manifest=fetch_manifest()
assert all(x in manifest.columns for x in ('asset_id','path','size'))
selected=[]
for mouse,aid,expected in PINNED:
    matches=manifest[manifest.asset_id==aid]
    if len(matches)!=1:raise RuntimeError(f'Pinned asset {aid} unavailable in published version')
    rec=matches.iloc[0]
    if int(rec['size'])!=expected: raise RuntimeError(f'Asset size mismatch: {mouse}')
    if not str(rec.path).startswith(mouse+'/'):raise RuntimeError(f'Subject mismatch: {mouse}')
    selected.append({'mouse':mouse,'asset_id':str(rec.asset_id),'path':str(rec.path),
                     'size_bytes':int(rec['size']),'size_MB':int(rec['size'])/1e6})
selection=pd.DataFrame(selected)
assert selection.mouse.nunique()==5
selection.to_csv(OUT/'pinned_selected_assets.csv',index=False)
print('Selected animals:',selection.mouse.nunique(), '| total download MB:',round(selection.size_MB.sum(),2))
display(selection[['mouse','size_MB','path']])

Selected animals: 5 | total download MB: 401.98


,mouse,size_MB,path
0,sub-anm266945,46.543184,sub-anm266945/sub-anm266945_ses-20141221_obj-1...
1,sub-anm266951,58.123096,sub-anm266951/sub-anm266951_ses-20141201_behav...
2,sub-anm281592,78.389912,sub-anm281592/sub-anm281592_ses-20150309_behav...
3,sub-anm244025,104.930672,sub-anm244025/sub-anm244025_ses-20141024_obj-1...
4,sub-anm260073,113.990856,sub-anm260073/sub-anm260073_ses-20141008_behav...


## 2. Download the five selected NWB assets (bounded)

The original archive is much larger. These five assets are processed sequentially. If the run is interrupted, already downloaded files in the active Kaggle session are reused.

In [3]:
def fetch_asset(row):
    aid=str(row.asset_id); expected=int(row.size_bytes)
    if not 0 < expected <= int(MAX_PER_FILE_MB*1e6):raise RuntimeError(f'Unsafe asset size {aid}')
    path=CACHE/(aid+'.nwb')
    if path.exists() and path.stat().st_size==expected:return path,0
    part=path.with_suffix('.part'); n=0
    try:
        with HTTP.get(f'{API}/api/assets/{aid}/download/',stream=True,timeout=(30,180)) as r:
            r.raise_for_status()
            with open(part,'wb') as f:
                for chunk in r.iter_content(1024*1024):
                    if not chunk:continue
                    n+=len(chunk)
                    if n>expected+4096 or n>MAX_PER_FILE_MB*1e6:raise RuntimeError('Exceeded download size cap')
                    f.write(chunk)
        if n!=expected:raise IOError(f'Incomplete file {aid}: {n} != {expected}')
        part.replace(path)
        return path,n
    finally:
        if part.exists():part.unlink()

FILES={}; transferred=0
for row in selection.itertuples(index=False):
    if transferred + int(row.size_bytes)>MAX_DOWNLOAD_MB*1e6:
        ERRORS.append({'mouse':row.mouse,'stage':'quota','error':'Exceeded total file cap'})
        continue
    try:
        path,bytes_new=fetch_asset(row)
        FILES[row.mouse]=path;transferred+=bytes_new
        print('Ready:',row.mouse,'MB:',round(path.stat().st_size/1e6,2))
    except Exception as exc:
        ERRORS.append({'mouse':row.mouse,'stage':'download','error':repr(exc)})
        print('Download failed:',row.mouse,repr(exc))
print('Available recordings',len(FILES),'bytes transferred MB',round(transferred/1e6,2))

Ready: sub-anm266945 MB: 46.54
Ready: sub-anm266951 MB: 58.12
Ready: sub-anm281592 MB: 78.39
Ready: sub-anm244025 MB: 104.93
Ready: sub-anm260073 MB: 113.99
Available recordings 5 bytes transferred MB 401.98


## 3. Safely extract time-aligned optical, voltage, and injected-current series

Voltage is converted to mV and injected current to pA, using NWB scaling metadata. Missing or incompatible current/voltage units are flagged—not silently assumed. All potential physical light-power interpretations remain unverified.

In [4]:
def decode(v):
    if isinstance(v,bytes):return v.decode('utf-8','replace')
    if isinstance(v,np.ndarray):
        return decode(v.reshape(-1)[0]) if v.size else ''
    return str(v)

def find_recordings(h):
    res={'optical':[],'voltage':[],'current':[]}
    def visit(path,obj):
        if not isinstance(obj,h5py.Group) or not isinstance(obj.get('data'),h5py.Dataset):return
        nt=decode(obj.attrs.get('neurodata_type','')).lower()
        if 'optogeneticseries' in nt:res['optical'].append(path)
        elif 'currentclampstimulusseries' in nt:res['current'].append(path)
        elif 'currentclampseries' in nt and path.lower().startswith('acquisition/'):
            res['voltage'].append(path)
    h.visititems(visit)
    return res

def read_time_series(group, cap=MAX_SERIES_SAMPLES):
    ds=group['data'];n=int(ds.shape[0])
    if n<4 or n>cap:raise ValueError(f'Unsafe data array length {n}')
    if ds.ndim==1:raw=np.asarray(ds[:],dtype=np.float64)
    elif ds.ndim==2 and ds.shape[1]==1:raw=np.asarray(ds[:,0],dtype=np.float64)
    else:raise ValueError('Multichannel array requires explicit channel selection')
    if isinstance(group.get('timestamps'),h5py.Dataset):
        ts=group['timestamps'];
        if len(ts)!=n:raise ValueError('NWB timestamps/data length mismatch')
        times=np.asarray(ts[:],dtype=np.float64); timing='timestamps'
    else:
        st=group.get('starting_time')
        if not isinstance(st,h5py.Dataset):raise ValueError('No timestamps or starting_time')
        t0=float(st[()]); rate=float(st.attrs.get('rate',np.nan))
        if not np.isfinite(rate) or rate<=0:raise ValueError('Invalid NWB sample rate')
        times=t0+np.arange(n)/rate;timing='rate'
    if not np.isfinite(times).all() or not np.all(np.diff(times)>0):raise ValueError('Invalid time axis')
    if not np.isfinite(raw).all():raise ValueError('Nonfinite recording data')
    conv=float(np.asarray(ds.attrs.get('conversion',1.)).reshape(-1)[0]);off=float(np.asarray(ds.attrs.get('offset',0.)).reshape(-1)[0])
    if not (np.isfinite(conv) and np.isfinite(off)):raise ValueError('Invalid NWB physical scaling')
    raw=raw*conv+off
    unit=decode(ds.attrs.get('unit',group.attrs.get('unit','unknown'))).strip().lower()
    return times,raw,unit,timing

def scale_voltage(x,unit):
    if unit in ('volts','volt','v'):return x*1e3
    if unit in ('mv','millivolt','millivolts'):return x
    raise ValueError(f'Unsupported voltage unit: {unit}')

def scale_current(x,unit):
    if unit in ('a','ampere','amperes','amp','amps'):return x*1e12
    if unit in ('pa','picoampere','picoamperes'):return x
    if unit in ('na','nanoampere','nanoamperes'):return x*1e3
    raise ValueError(f'Unsupported current unit: {unit}')

def get_command_pulses(t,v,mouse,site):
    # A candidate pulse is a positive-then-zero transition, not proof that light reached tissue.
    if len(np.unique(v))>100 or not (np.any(v==0) and np.any(v>0)):
        raise ValueError('Optical series cannot be interpreted as positive-zero command')
    active=v>0
    starts=np.flatnonzero(~active[:-1]&active[1:])+1
    ends=np.flatnonzero(active[:-1]&~active[1:])+1
    if len(starts)>MAX_PULSES_PER_SERIES:raise ValueError('Optical series exceeds full-extraction cap; no partial analysis')
    result=[]
    for a in starts:
        ix=np.searchsorted(ends,a,side='right')
        if ix>=len(ends):continue
        b=int(ends[ix]);start=float(t[a]);end=float(t[b])
        if end>start:
            result.append({'mouse':mouse,'site':site,'onset_s':start,'offset_s':end,
                'duration_ms':(end-start)*1e3,'raw_optical_value_UNCALIBRATED':float(v[a])})
    return result

## 4. Recover complete pulse trains and audit train-definition sensitivity

Primary train definition: pulses with **at most 200 ms between the previous pulse end and next pulse start** are grouped into one train. We also report train counts for alternative gaps so conclusions do not depend on one arbitrary threshold.

Some short-pulse records have 5-ms pulses at approximately 10-ms onset intervals; individual pulses are **not** treated as independent experiments.

In [5]:
def make_trains(pulses,gap_ms=200.0):
    if pulses.empty:return pd.DataFrame()
    p=pulses.sort_values(['onset_s','offset_s','site']).reset_index(drop=True)
    groups=[];start=0;current_end=float(p.loc[0,'offset_s'])
    for i in range(1,len(p)):
        a=float(p.loc[i,'onset_s']);b=float(p.loc[i,'offset_s'])
        if a-current_end > gap_ms/1e3:
            groups.append(p.iloc[start:i]);start=i;current_end=b
        else:current_end=max(current_end,b)
    groups.append(p.iloc[start:])
    rows=[]
    for k,g in enumerate(groups):
        t0=float(g.onset_s.min());t1=float(g.offset_s.max())
        dur_ms=(t1-t0)*1e3
        pulse_dur=float(g.duration_ms.median())
        if len(g)>1 and pulse_dur<25: family='short_pulse_train'
        elif len(g)==1 and pulse_dur>=250: family='isolated_long_pulse'
        elif len(g)==1 and pulse_dur<25: family='single_short_pulse'
        else:family='other_or_mixed'
        ggap=(g.onset_s.iloc[1:].to_numpy()-g.offset_s.iloc[:-1].to_numpy())*1000 if len(g)>1 else []
        rows.append({'mouse':str(g.mouse.iloc[0]),'train_id':f'{g.mouse.iloc[0]}_T{k:03d}',
            'train_start_s':t0,'train_end_s':t1,'train_duration_ms':dur_ms,
            'n_pulses':int(len(g)),'median_pulse_ms':pulse_dur,
            'mean_interpulse_gap_ms':float(np.mean(ggap)) if len(ggap) else np.nan,
            'n_stimulation_sites':int(g.site.nunique()),'sites':' | '.join(sorted(g.site.unique())),
            'protocol_family':family,'optical_codes_UNCALIBRATED':' | '.join(sorted(map(str,g.raw_optical_value_UNCALIBRATED.unique())))})
    return pd.DataFrame(rows)


def timed_window(t,v,start,end,minimum=4):
    if not (np.isfinite(start) and np.isfinite(end) and end>start):return np.array([])
    a=int(np.searchsorted(t,start));b=int(np.searchsorted(t,end))
    if b-a<minimum:return np.array([])
    tt=t[a:b];vv=v[a:b];dt=np.median(np.diff(tt))
    if dt<=0 or np.max(np.diff(tt))>max(0.015,10*dt):return np.array([])
    if tt[0]-start>max(0.008,5*dt) or end-tt[-1]>max(0.008,5*dt):return np.array([])
    return vv


def v_measures(t,v,start,end):
    # Comparisons are means in mV and intervals always relative to entire optical train.
    pre=timed_window(t,v,start-.20,start-.02)
    early=timed_window(t,v,start+.005,start+.05)
    late=timed_window(t,v,start+.05,start+.15)
    within=timed_window(t,v,start+.05,end-.02) if end-start>.10 else np.array([])
    post=timed_window(t,v,end+.02,end+.20)
    if not (len(pre) and len(early) and len(post)):
        return {'complete_voltage':False,'baseline_mV':np.nan,'early_delta_mV':np.nan,
          'late_delta_mV':np.nan,'within_train_delta_mV':np.nan,'post_delta_mV':np.nan,
          'voltage_baseline_sd_mV':np.nan}
    m=float(np.mean(pre))
    return {'complete_voltage':True,'baseline_mV':m,
        'early_delta_mV':float(np.mean(early)-m),
        'late_delta_mV':float(np.mean(late)-m) if len(late) else np.nan,
        'within_train_delta_mV':float(np.mean(within)-m) if len(within) else np.nan,
        'post_delta_mV':float(np.mean(post)-m),
        'voltage_baseline_sd_mV':float(np.std(pre))}


def i_measures(t,i,start,end):
    pre=timed_window(t,i,start-.20,start-.02)
    early=timed_window(t,i,start+.005,start+.05)
    post=timed_window(t,i,end+.02,end+.20)
    # Sample the full ON period to catch confounding changes mid-train.
    on=timed_window(t,i,start,end)
    if not (len(pre) and len(early) and len(post) and len(on)):
        return {'current_qc':'unresolved','baseline_current_pA':np.nan,
            'on_current_mean_pA':np.nan,'max_abs_on_current_change_pA':np.nan,
            'early_current_change_pA':np.nan,'post_current_change_pA':np.nan}
    base=float(np.median(pre))
    noise=float(median_abs_deviation(pre,scale='normal'))
    th=max(5.0,6*noise) # 5 pA absolute floor, or 6 times measured baseline fluctuation
    devmax=float(np.max(np.abs(on-base)))
    delta=float(np.mean(early)-base); postdelta=float(np.mean(post)-base)
    flag='possible_current_step' if max(devmax,abs(delta),abs(postdelta))>th else 'no_large_step_detected'
    return {'current_qc':flag,'baseline_current_pA':base,'on_current_mean_pA':float(np.mean(on)),
            'max_abs_on_current_change_pA':devmax,
            'early_current_change_pA':delta,'post_current_change_pA':postdelta,
            'current_change_threshold_pA':th}


def train_level_response(trains,vt,vmv,it=None,ipa=None):
    res=[]
    for r in trains.itertuples(index=False):
        d=r._asdict();s=float(d['train_start_s']);e=float(d['train_end_s'])
        d.update(v_measures(vt,vmv,s,e))
        d.update(i_measures(it,ipa,s,e) if it is not None else {'current_qc':'unresolved'})
        d['qc_pass']=bool(d['complete_voltage'] and d['current_qc']=='no_large_step_detected')
        res.append(d)
    return pd.DataFrame(res)

## 5. Match non-stimulation (sham) windows while holding the current-injection state similar

For each quality-controlled train, random non-stimulation windows of the **same length** are sampled from the same mouse, far from recorded optical commands. We require similar baseline injected current and baseline voltage and reject current steps. This is an **exploratory timing control**, not a randomized experiment.

In [6]:
def make_matched_shams(rows,trains,vt,vmv,it,ipa,n_each=SHAM_PER_TRAIN):
    if len(rows)==0:return pd.DataFrame()
    starts=np.sort(trains.train_start_s.to_numpy())
    ends=trains.sort_values('train_start_s').train_end_s.to_numpy()
    result=[]
    lo=float(max(vt[0],it[0] if it is not None else vt[0])+.25)
    hi=float(min(vt[-1],it[-1] if it is not None else vt[-1])-.25)
    for row in rows.itertuples(index=False):
        if not row.qc_pass:continue
        duration=float(row.train_end_s-row.train_start_s)
        if hi-lo<duration+.1:continue
        found=0
        for _ in range(SHAM_MAX_ATTEMPTS_PER_TRAIN):
            if found>=n_each:break
            s=float(rng.uniform(lo,hi-duration));e=s+duration
            idx=np.searchsorted(starts,s,side='right')
            prev_end=float(ends[idx-1]) if idx else -np.inf
            next_start=float(starts[idx]) if idx<len(starts) else np.inf
            if not (s-prev_end>=MIN_SHAM_DISTANCE_S and next_start-e>=MIN_SHAM_DISTANCE_S):continue
            vm=v_measures(vt,vmv,s,e)
            if not vm['complete_voltage']:continue
            if abs(vm['baseline_mV']-float(row.baseline_mV))>10.0:continue
            im=i_measures(it,ipa,s,e) if it is not None else {'current_qc':'unresolved'}
            if im['current_qc']!='no_large_step_detected':continue
            tol=max(10.0,abs(float(row.baseline_current_pA))*0.20)
            if abs(im['baseline_current_pA']-float(row.baseline_current_pA))>tol:continue
            result.append({'mouse':row.mouse,'train_id':row.train_id,'sham_id':found,
               'sham_start_s':s,'sham_end_s':e,'sham_early_delta_mV':vm['early_delta_mV'],
               'sham_post_delta_mV':vm['post_delta_mV'],
               'sham_baseline_mV':vm['baseline_mV'],'sham_baseline_current_pA':im['baseline_current_pA']})
            found+=1
    return pd.DataFrame(result)


def within_mouse_summary(trains,shams):
    if trains.empty:return pd.DataFrame()
    summaries=[]
    gshams=shams.groupby('train_id') if len(shams) else None
    for mouse,g in trains.groupby('mouse'):
        clean=g[g.qc_pass].copy()
        rec={'mouse':mouse,'n_clean_trains':int(len(clean)),
             'mean_true_early_delta_mV':float(clean.early_delta_mV.mean()) if len(clean) else np.nan,
             'mean_true_post_delta_mV':float(clean.post_delta_mV.mean()) if len(clean) else np.nan}
        dif=[];matched=0
        for row in clean.itertuples(index=False):
            if gshams is not None and row.train_id in gshams.groups:
                h=gshams.get_group(row.train_id)
                if len(h)>=5:
                    dif.append(float(row.early_delta_mV-h.sham_early_delta_mV.mean()));matched+=1
        rec['n_trials_with_at_least_5_matched_shams']=matched
        rec['mean_paired_true_minus_sham_early_mV']=float(np.mean(dif)) if dif else np.nan
        if len(dif)>=8:
            bs=[float(np.mean(rng.choice(dif,size=len(dif),replace=True))) for _ in range(1000)]
            rec['within_mouse_bootstrap_95CI_low_mV'],rec['within_mouse_bootstrap_95CI_high_mV']=map(float,np.quantile(bs,[.025,.975]))
        else:
            rec['within_mouse_bootstrap_95CI_low_mV']=np.nan
            rec['within_mouse_bootstrap_95CI_high_mV']=np.nan
        summaries.append(rec)
    return pd.DataFrame(summaries)

## 6. Process real recordings and export measured responses

This cell can take several minutes because full time series must be read. It generates the train hierarchy, evaluates injected-current changes, and saves real NWB voltage/current examples. It does **not** fabricate missing values.

In [7]:
ALL_PULSES=[];ALL_TRAINS=[];ALL_RESPONSES=[];ALL_SHAMS=[];ALL_SERIES=[];SENS=[];MOUSE_SUMMARY=[]
for row in selection.itertuples(index=False):
    mouse=row.mouse
    if mouse not in FILES: continue
    filepath=FILES[mouse]
    try:
        with h5py.File(filepath,'r') as h:
            groups=find_recordings(h)
            if not groups['optical'] or not groups['voltage']:
                raise ValueError('Missing optical or CurrentClampSeries')
            if len(groups['voltage'])>1:raise ValueError('Multiple voltage recordings; need manual mapping')
            vt,v,vu,v_mode=read_time_series(h[groups['voltage'][0]])
            vmv=scale_voltage(v,vu)
            current=None
            if len(groups['current'])==1:
                it,iv,iu,i_mode=read_time_series(h[groups['current'][0]])
                ipa=scale_current(iv,iu);current=(it,ipa)
            else:it=ipa=None;i_mode='missing_or_multiple';iu='unknown'
            pulses=[]
            for name in groups['optical']:
                try:
                    ot,ov,ou,o_mode=read_time_series(h[name])
                    obs=get_command_pulses(ot,ov,mouse,name)
                    pulses.extend(obs)
                    ALL_SERIES.append({'mouse':mouse,'type':'optical','name':name,'n_samples':len(ot),
                         'n_events':len(obs),'unit_stored':ou,'time_mode':o_mode,
                         'start_s':float(ot[0]),'end_s':float(ot[-1])})
                except Exception as exc:
                    ERRORS.append({'mouse':mouse,'stage':'optical_series','error':f'{name}: {repr(exc)}'})
            ALL_SERIES.append({'mouse':mouse,'type':'voltage','name':groups['voltage'][0],
                'n_samples':len(vt),'n_events':0,'unit_stored':vu,'time_mode':v_mode,
                'start_s':float(vt[0]),'end_s':float(vt[-1])})
            ALL_SERIES.append({'mouse':mouse,'type':'current','name':str(groups['current']),
                'n_samples':len(it) if it is not None else 0,'n_events':0,'unit_stored':iu,'time_mode':i_mode,
                'start_s':float(it[0]) if it is not None else np.nan,
                'end_s':float(it[-1]) if it is not None else np.nan})
            if not pulses:raise ValueError('No valid candidate optical pulses')
            p=pd.DataFrame(pulses).sort_values('onset_s').reset_index(drop=True)
            p['asset_id']=row.asset_id
            train=make_trains(p,TRAIN_GAP_MS)
            if train.empty:raise ValueError('No stimulation trains')
            for gap in GAP_SENSITIVITY_MS:
                q=make_trains(p,gap)
                SENS.append({'mouse':mouse,'gap_ms':gap,'n_trains':len(q),'n_multi_pulse_trains':int((q.n_pulses>1).sum()),
                             'max_pulses_in_one_train':int(q.n_pulses.max())})
            response=train_level_response(train,vt,vmv,it,ipa)
            sham=make_matched_shams(response,train,vt,vmv,it,ipa)
            ALL_PULSES.append(p);ALL_TRAINS.append(train);ALL_RESPONSES.append(response)
            if len(sham):ALL_SHAMS.append(sham)
            MOUSE_SUMMARY.append({'mouse':mouse,'status':'success','n_individual_pulses':len(p),
                'n_200ms_trains':len(train),'n_train_qc_pass':int(response.qc_pass.sum()),
                'n_current_step_flagged':int((response.current_qc=='possible_current_step').sum()),
                'n_current_unresolved':int((response.current_qc=='unresolved').sum()),
                'n_with_match_shams':int(sham.train_id.nunique()) if len(sham) else 0,
                'n_shams':int(len(sham)),'median_pulse_duration_ms':float(p.duration_ms.median()),
                'protocol_families':' | '.join(sorted(train.protocol_family.unique()))})
            # Plot first few candidate train responses directly from recorded NWB data.
            fig,axs=plt.subplots(2,1,figsize=(9,5.6),sharex=True)
            ex=response.iloc[0]
            lo=ex.train_start_s-.25;hi=min(ex.train_end_s+.25,ex.train_start_s+6.0)
            m=(vt>=lo)&(vt<=hi)
            axs[0].plot(vt[m]-ex.train_start_s,vmv[m],lw=.7)
            axs[0].set(ylabel='Voltage (mV)',title=f'{mouse}: actual NWB signals, first complete train')
            if it is not None:
                mm=(it>=lo)&(it<=hi)
                axs[1].plot(it[mm]-ex.train_start_s,ipa[mm],lw=.7)
                axs[1].set_ylabel('Injected current (pA)')
            else:axs[1].text(.3,.5,'Current unavailable',transform=axs[1].transAxes)
            for ax in axs:
                ax.axvline(0,ls='--',c='black',lw=.7)
                ax.axvspan(0,min(hi-ex.train_start_s,ex.train_end_s-ex.train_start_s),alpha=.12,color='gray')
            axs[1].set_xlabel('Time from candidate optical train start (s)')
            fig.tight_layout();fig.savefig(OUT/f'{mouse}_voltage_current_train_example.png',dpi=165);plt.close(fig)
            print(mouse,':',len(p),'pulses ->',len(train),'trains; QC',int(response.qc_pass.sum()),
                  'train-level shams',sham.train_id.nunique() if len(sham) else 0)
    except Exception as exc:
        ERRORS.append({'mouse':mouse,'stage':'process_mouse','error':repr(exc)+' '+traceback.format_exc(limit=2)})
        MOUSE_SUMMARY.append({'mouse':mouse,'status':'FAILED','error':repr(exc)})
        print('FAILED',mouse,repr(exc))
    finally:
        if DELETE_RAW_AFTER_ANALYSIS and filepath.exists():filepath.unlink()

P=pd.concat(ALL_PULSES,ignore_index=True) if ALL_PULSES else pd.DataFrame()
T=pd.concat(ALL_TRAINS,ignore_index=True) if ALL_TRAINS else pd.DataFrame()
R=pd.concat(ALL_RESPONSES,ignore_index=True) if ALL_RESPONSES else pd.DataFrame()
H=pd.concat(ALL_SHAMS,ignore_index=True) if ALL_SHAMS else pd.DataFrame(columns=['mouse','train_id','sham_id'])
S=pd.DataFrame(MOUSE_SUMMARY)
F=pd.DataFrame(SENS)
W=pd.DataFrame(ALL_SERIES)
for name,d in [('candidate_light_pulses_UNCALIBRATED.csv',P),('protocol_aware_train_events.csv',T),
 ('train_level_voltage_current_responses_EXPLORATORY.csv',R),('matched_sham_responses_EXPLORATORY.csv',H),
 ('mouse_feasibility.csv',S),('train_gap_sensitivity.csv',F),('recording_timebase_inventory.csv',W)]:
    d.to_csv(OUT/name,index=False)
pd.DataFrame(ERRORS,columns=['mouse','stage','error']).to_csv(OUT/'errors_and_warnings.csv',index=False)
display(S)

sub-anm266945 : 5 pulses -> 5 trains; QC 4 train-level shams 4
sub-anm266951 : 5 pulses -> 5 trains; QC 4 train-level shams 4
sub-anm281592 : 298 pulses -> 5 trains; QC 5 train-level shams 5
sub-anm244025 : 46 pulses -> 46 trains; QC 37 train-level shams 37
sub-anm260073 : 3500 pulses -> 10 trains; QC 3 train-level shams 2


,mouse,status,n_individual_pulses,n_200ms_trains,n_train_qc_pass,n_current_step_flagged,n_current_unresolved,n_with_match_shams,n_shams,median_pulse_duration_ms,protocol_families
0,sub-anm266945,success,5,5,4,1,0,4,66,1499.9,isolated_long_pulse
1,sub-anm266951,success,5,5,4,1,0,4,70,999.9,isolated_long_pulse
2,sub-anm281592,success,298,5,5,0,0,5,150,5.0,short_pulse_train
3,sub-anm244025,success,46,46,37,9,0,37,659,500.0,isolated_long_pulse
4,sub-anm260073,success,3500,10,3,7,0,2,7,5.0,short_pulse_train


## 7. Compare true optical-command times with matched sham times

These contrasts remain observational and are reported with *train-level* uncertainty within each mouse. No cross-animal statistical significance is claimed from repeated events within a mouse.

In [8]:
X=within_mouse_summary(R,H)
X.to_csv(OUT/'within_mouse_true_vs_sham_summary.csv',index=False)
if len(R):
    fig,ax=plt.subplots(figsize=(9,4.4))
    if len(T):
        s=S[S.status=='success'].copy()
        ax.bar(s.mouse.str.replace('sub-','',regex=False),s.n_individual_pulses,alpha=.4,label='Individual pulse commands')
        ax.bar(s.mouse.str.replace('sub-','',regex=False),s.n_200ms_trains,label='200-ms-separated trains')
        ax.set_yscale('symlog',linthresh=3)
    ax.set(ylabel='Number of observations (symlog)',title='Pulse versus train sample size: five real mice')
    ax.tick_params(axis='x',rotation=20)
    ax.legend();fig.tight_layout();fig.savefig(OUT/'fig_pulses_versus_independent_trains.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4.3))
    colors=['C0','C1','C2','C3','C4']
    for i,(mouse,g) in enumerate(R.groupby('mouse')):
        sub=g[g.qc_pass]
        if not sub.empty:
            ax.scatter(np.full(len(sub),i),sub.early_delta_mV,s=12,alpha=.45,color=colors[i%len(colors)])
            ax.plot(i,sub.early_delta_mV.mean(),'D',color='black',markersize=6)
    ax.axhline(0,color='black',lw=.75,ls='--')
    ax.set_xticks(range(R.mouse.nunique()),[m.replace('sub-','') for m in sorted(R.mouse.unique())],rotation=20)
    ax.set(ylabel='Early voltage delta from baseline (mV)',title='Exploratory train-level response distribution by mouse')
    fig.tight_layout();fig.savefig(OUT/'fig_train_level_early_voltage_response.png',dpi=180);plt.close(fig)
print('Matched sham outcomes (within mouse only):')
display(X)

Matched sham outcomes (within mouse only):


,mouse,n_clean_trains,mean_true_early_delta_mV,mean_true_post_delta_mV,n_trials_with_at_least_5_matched_shams,mean_paired_true_minus_sham_early_mV,within_mouse_bootstrap_95CI_low_mV,within_mouse_bootstrap_95CI_high_mV
0,sub-anm244025,37,-0.084464,-0.012726,34,-0.075919,-0.12073,-0.028713
1,sub-anm260073,3,-3.658041,1.007908,1,-2.994720,NaN,NaN
2,sub-anm266945,4,1.484275,-3.029086,4,0.877413,NaN,NaN
3,sub-anm266951,4,-2.643293,-0.046594,4,-2.326163,NaN,NaN
4,sub-anm281592,5,0.446782,0.259857,5,0.443827,NaN,NaN


## 8. Pre-specified biological training gate

**Conservative GO rule** (all must hold): At least 3 independently recorded mice under closely comparable pulse *and train* protocols, each with at least 10 quality-controlled trains; reliable voltage/current timing and stable injected current; independent optical calibration and verified response alignment; measurable train-specific response separating optical commands from matched sham periods in independently held-out mice.

No amount of extra repeated pulses from one animal satisfies independent-mouse replication. Failure of the GO rule is a **dataset-suitability conclusion**, not evidence that optogenetics is ineffective.

In [9]:
# Report the evidence instead of declaring causality from low mouse count.
mouse_success=S[S.status=='success'].copy() if len(S) and 'status' in S.columns else pd.DataFrame()
protocol_counts = (R[R.qc_pass].groupby(['protocol_family','mouse']).size().reset_index(name='clean_trains')
                   if len(R) else pd.DataFrame(columns=['protocol_family','mouse','clean_trains']))
protocol_counts.to_csv(OUT/'protocol_compatible_train_counts.csv',index=False)
qualifying=[]
for family,group in protocol_counts.groupby('protocol_family'):
    mice=int((group.clean_trains>=10).sum())
    qualifying.append({'protocol_family':family,'mice_with_10_clean_trains':mice,
       'at_least_3_qualifying_mice':mice>=3})
Q=pd.DataFrame(qualifying)
Q.to_csv(OUT/'protocol_family_eligibility.csv',index=False)

# Protocol families here are broad; a third-stage duration/dose/calibration audit would still be necessary.
count_gate=bool(len(Q) and Q.at_least_3_qualifying_mice.any())
science_gate=False # No independent illumination calibration or causal response verification is in this NWB-only audit.
summary={
 'dataset_id':DANDI_ID,'version':VERSION,'analysis_utc':datetime.now(timezone.utc).isoformat(),
 'mice_requested':len(selection),'mice_successfully_processed':len(mouse_success),
 'candidate_optical_pulse_commands':int(len(P)),
 'protocol_aware_200ms_trains':int(len(T)),
 'train_level_qc_pass':int(R.qc_pass.sum()) if len(R) else 0,
 'mice_with_at_least_10_qc_trains':int((mouse_success.n_train_qc_pass>=10).sum()) if len(mouse_success) else 0,
 'protocol_count_gate':count_gate,'independent_light_calibration_verified':False,
 'biophysical_causation_established':False,'safe_for_cross_animal_DL_training':bool(count_gate and science_gate),
 'recommendation':('Continue methodological investigation, but do NOT start Q1-level cross-animal optogenetic DL on these files alone. '
                   'Expand truly comparable biological recordings or pivot dataset if the gate remains unmet.'),
 'limitations':['Only five mouse recordings inspected',
     'Optical command values were not independently power-calibrated',
     'Injected-current trace is a confounder even if large steps are excluded',
     'Sham controls and within-mouse bootstrap do not establish causation or cross-animal generalization',
     'Mouse cortical/thalamic data cannot validate a human epilepsy digital twin']}
(OUT/'phase2dd_go_no_go.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
display(Q)

{
  "dataset_id": "000005",
  "version": "0.220126.1853",
  "analysis_utc": "2026-10-13T16:04:14.449664+00:00",
  "mice_requested": 5,
  "mice_successfully_processed": 5,
  "candidate_optical_pulse_commands": 3854,
  "protocol_aware_200ms_trains": 71,
  "train_level_qc_pass": 53,
  "mice_with_at_least_10_qc_trains": 1,
  "protocol_count_gate": false,
  "independent_light_calibration_verified": false,
  "biophysical_causation_established": false,
  "safe_for_cross_animal_DL_training": false,
  "recommendation": "Continue methodological investigation, but do NOT start Q1-level cross-animal optogenetic DL on these files alone. Expand truly comparable biological recordings or pivot dataset if the gate remains unmet.",
  "limitations": [
    "Only five mouse recordings inspected",
    "Optical command values were not independently power-calibrated",
    "Injected-current trace is a confounder even if large steps are excluded",
    "Sham controls and within-mouse bootstrap do not establish c

,protocol_family,mice_with_10_clean_trains,at_least_3_qualifying_mice
0,isolated_long_pulse,1,False
1,short_pulse_train,0,False


## 9. Export results ZIP

Includes raw-data-derived metrics and figures; original NWB files are deliberately excluded. **Please upload the resulting ZIP for review.**

In [10]:
readme=f"""Phase 2D-D protocol-aware response audit (DANDI {DANDI_ID} version {VERSION}).
REAL NWB analysis only. Generated UTC: {datetime.now(timezone.utc).isoformat()}

Interpretation:
- Each row in protocol_aware_train_events.csv is an aggregated optical COMMAND train (gap <= {TRAIN_GAP_MS} ms).
- Physical light delivery and absolute optical intensity were not independently verified.
- train_level_voltage_current_responses_EXPLORATORY.csv contains voltage differences and injected-current QC.
- matched_sham_responses_EXPLORATORY.csv contains within-mouse candidate control windows, not biological counterfactuals.
- Within-mouse bootstrap is exploratory and DOES NOT imply across-animal generalization.
- If more than one optic site contains overlapping pulses, inspect sites before causal conclusions.
- Full DL training is intentionally gated by independent mice, matched protocols, and optical validation.
"""
(OUT/'READ_ME_FIRST.txt').write_text(readme)
zip_path=Path('/kaggle/working/phase2dd_protocol_aware_results.zip') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2dd_protocol_aware_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for p in sorted(OUT.glob('*')):
        if p.is_file() and p.suffix.lower() in ('.csv','.png','.json','.txt'):
            z.write(p,arcname=p.name)
print('RESULT ZIP:',zip_path,'size MB:',round(zip_path.stat().st_size/1e6,2))

RESULT ZIP: /kaggle/working/phase2dd_protocol_aware_results.zip size MB: 0.75
